In [21]:
from coevolution.analysis.log_parser import parse_coevolution_log
from coevolution.analysis.lineage import build_lineage_graph, get_ancestral_subgraph

results = parse_coevolution_log(
    '../../../logs',
    "*med-missing*",
    'med-missing',
    'abc398_b'
)
ind_df = results['individuals']

code_ind_df = ind_df[ind_df['type'] == 'code']
survived_df = code_ind_df.loc[(code_ind_df['status']=='SURVIVED')]
survived_df

2026-01-26 23:11:19.293 | INFO     | coevolution.analysis.log_parser:_log_line_generator:51 - Streaming: coevolution_run_20260125_med-missing_MASTER.log
2026-01-26 23:11:19.343 | INFO     | coevolution.analysis.log_parser:parse_coevolution_log:180 - Constructing DataFrames...
2026-01-26 23:11:19.346 | SUCCESS  | coevolution.analysis.log_parser:parse_coevolution_log:194 - Parsing Complete. Found matrix types: ['private', 'differential', 'unittest', 'public']


,id,type,snippet,creation_op,generation_born,probability,parents,metadata,lifecycle_events,status,run_id
102,C11,code,"class Solution:\n def sol(self, input_str: ...",crossover,1,0.9976,"{'code': ['C9', 'C3'], 'test': []}",{'operation': 'crossover'},"[{'generation': 1, 'event': 'created', 'detail...",SURVIVED,med-missing
103,C4,code,"class Solution:\n def sol(self, input_str: ...",initial,0,0.8945,"{'code': [], 'test': []}",{},"[{'generation': 0, 'event': 'created', 'detail...",SURVIVED,med-missing
104,C8,code,"class Solution:\n def sol(self, input_str: ...",initial,0,0.7156,"{'code': [], 'test': []}",{},"[{'generation': 0, 'event': 'created', 'detail...",SURVIVED,med-missing
105,C22,code,"class Solution:\n def sol(self, input_str: ...",mutation,4,0.9999,"{'code': ['C14'], 'test': []}",{'operation': 'mutation'},"[{'generation': 4, 'event': 'created', 'detail...",SURVIVED,med-missing
106,C12,code,"class Solution:\n def sol(self, input_str: ...",crossover,1,0.9893,"{'code': ['C0', 'C1'], 'test': []}",{'operation': 'crossover'},"[{'generation': 1, 'event': 'created', 'detail...",SURVIVED,med-missing
107,C1,code,"class Solution:\n def sol(self, input_str: ...",initial,0,0.6349,"{'code': [], 'test': []}",{},"[{'generation': 0, 'event': 'created', 'detail...",SURVIVED,med-missing
108,C15,code,"class Solution:\n def sol(self, input_str: ...",edit,2,0.9936,"{'code': ['C8'], 'test': ['T88', 'T84', 'T107'...","{'operation': 'edit', 'num_failing_tests': 10,...","[{'generation': 2, 'event': 'created', 'detail...",SURVIVED,med-missing
109,C24,code,"class Solution:\n def sol(self, input_str: ...",crossover,4,0.9994,"{'code': ['C11', 'C16'], 'test': []}",{'operation': 'crossover'},"[{'generation': 4, 'event': 'created', 'detail...",SURVIVED,med-missing
110,C17,code,"class Solution:\n def sol(self, input_str: ...",edit,2,0.9968,"{'code': ['C4'], 'test': ['T110', 'T87', 'T83'...","{'operation': 'edit', 'num_failing_tests': 6, ...","[{'generation': 2, 'event': 'created', 'detail...",SURVIVED,med-missing
111,C25,code,"class Solution:\n def sol(self, input_str: ...",edit,4,1.0000,"{'code': ['C14'], 'test': ['T136', 'T141', 'T1...","{'operation': 'edit', 'num_failing_tests': 9, ...","[{'generation': 4, 'event': 'created', 'detail...",SURVIVED,med-missing


In [33]:
graph = build_lineage_graph(code_ind_df)
subgraph = get_ancestral_subgraph(graph, 
                                  'C24')

2026-01-26 23:23:27.418 | INFO     | coevolution.analysis.lineage:build_lineage_graph:27 - Building Lineage Graph...
2026-01-26 23:23:27.421 | SUCCESS  | coevolution.analysis.lineage:build_lineage_graph:71 - Graph created: 61 individuals, 127 lineage connections.


Tracing lineage for C24: Reduced 61 nodes -> 12 relevant ancestors.


In [34]:
from matplotlib import gridspec
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
from loguru import logger
import numpy as np
import seaborn as sns
from matplotlib.colors import ListedColormap
from adjustText import adjust_text
from coevolution.analysis.log_parser import parse_coevolution_log

import networkx as nx
from pyvis.network import Network
import os

def visualize_lineage_interactive(
    G: nx.DiGraph, 
    output_file: str = "coevolution_lineage.html"
) -> None:
    """
    Generates a high-performance interactive graph with explicit 'Zipper' layout.
    
    Fixes:
    - "Alternating View": Manually places Code vertically between Test generations.
    """
    if G is None or G.number_of_nodes() == 0:
        print("Graph is empty, nothing to visualize.")
        return

    print(f"Generating optimized interactive graph to {output_file}...")

    # --- 1. Initialize PyVis (No Physics/Layout Presets) ---
    # We turn off the 'hierarchical' preset because we are doing it manually.
    net = Network(height="95vh", width="100%", bgcolor="#ffffff", font_color="black", directed=True)
    
    # Strictly disable physics to prevent browser crash
    net.set_options("""
    var options = {
      "physics": {
        "enabled": false
      },
      "interaction": {
        "dragNodes": true,
        "hideEdgesOnDrag": false,
        "hideNodesOnDrag": false
      },
      "edges": {
        "smooth": {
          "type": "cubicBezier",
          "forceDirection": "vertical",
          "roundness": 0.4
        }
      }
    }
    """)

    # --- 2. Calculate Swimlane Widths ---
    grouped_nodes = defaultdict(lambda: defaultdict(list))
    max_width_per_type = defaultdict(int)
    
    for node, data in G.nodes(data=True):
        pop_type = data.get('type', 'unknown').lower()
        gen = data.get('gen_born', -1)
        if gen == -1: gen = 0
        grouped_nodes[pop_type][gen].append(node)
        # Track max density to size the lanes
        max_width_per_type[pop_type] = max(max_width_per_type[pop_type], len(grouped_nodes[pop_type][gen]))

    # --- 3. Define Layout Constants ---
    # X-Axis Spacing
    NODE_WIDTH = 100       # Horizontal space per node
    LANE_PADDING = 300     # Gap between population types
    
    # Y-Axis Spacing (The Zipper)
    GEN_HEIGHT = 400       # Vertical space for one full generation step
    CODE_OFFSET = 200      # How far down Code sits relative to Tests of same gen

    # Calculate Lane Centers
    current_x = 0
    lane_centers = {}
    
    # Order: Tests Left, Code Right (Visual flow: Tests -> Code)
    ordered_types = ['unittest', 'differential', 'public', 'code', 'unknown']
    
    for p_type in ordered_types:
        if p_type not in max_width_per_type: continue
        lane_width = max_width_per_type[p_type] * NODE_WIDTH
        lane_centers[p_type] = current_x + (lane_width / 2)
        current_x += lane_width + LANE_PADDING

    # --- 4. Place Nodes (Manual X/Y) ---
    for node, data in G.nodes(data=True):
        pop_type = data.get('type', 'unknown').lower()
        gen = data.get('gen_born', 0)
        status = data.get('status', 'Unknown')
        
        # --- CALCULATE Y (The Interleaved Logic) ---
        base_y = gen * GEN_HEIGHT
        
        if 'code' in pop_type:
            # Code is pushed down by half a step
            y_pos = base_y + CODE_OFFSET
        else:
            # Tests sit at the "top" of the generation
            y_pos = base_y

        # --- CALCULATE X (Centering) ---
        nodes_in_row = grouped_nodes[pop_type][gen]
        # Sort by ID to ensure consistent placement
        sorted_nodes = sorted(nodes_in_row)
        try:
            idx = sorted_nodes.index(node)
        except ValueError:
            idx = 0
            
        lane_center = lane_centers.get(pop_type, 0)
        row_width = (len(nodes_in_row) - 1) * NODE_WIDTH
        start_x = lane_center - (row_width / 2)
        x_pos = start_x + (idx * NODE_WIDTH)

        # --- STYLING ---
        color = "#97c2fc"
        shape = "dot"
        size = 20  # Default size
        
        if 'code' in pop_type:
            color = "#3498db" # Blue
            shape = "dot"
            size = 25
        elif 'unittest' in pop_type:
            color = "#2ecc71" # Green
            shape = "square"
        elif 'differential' in pop_type:
            color = "#e67e22" # Orange
            shape = "triangle"
            size = 15 # Slightly smaller to handle density
        elif 'public' in pop_type:
            color = "#9b59b6" # Purple
            shape = "diamond"

        # Highlight Survivors
        border_width = 1
        if status == 'SURVIVED':
            border_width = 3
            # Make survivors slightly larger
            size += 5
        else:
            # Fade out dead nodes slightly
            color = f"{color}80" # Add hex transparency (approximate support)

        # Tooltip
        title = (
            f"<b>{node}</b><br>"
            f"Type: {pop_type}<br>"
            f"Gen: {gen}<br>"
            f"Status: {status}<br>"
            f"Prob: {data.get('prob', 0.0):.4f}"
        )

        net.add_node(
            node,
            label=str(node),
            title=title,
            color=color,
            shape=shape,
            size=size,
            borderWidth=border_width,
            x=x_pos, 
            y=y_pos,
            physics=False,  # CRITICAL: Disable physics per node
            fixed=True      # CRITICAL: Lock position so they don't drift
        )

    # --- 5. Add Edges ---
    for u, v, data in G.edges(data=True):
        p_type = data.get('parent_type', 'unknown')
        target_status = G.nodes[v].get('status', 'Unknown')
        
        color = '#bdc3c7' # Default faint gray
        width = 1
        
        # Color by parent type
        if p_type == 'code': color = '#3498db'
        elif p_type == 'test': color = '#2ecc71'
        
        # Emphasize Survivor Paths
        if target_status == 'SURVIVED':
            width = 3
            # Keep original color
        else:
            # Fade dead edges significantly
            color = 'rgba(200, 200, 200, 0.3)'

        net.add_edge(
            u, v,
            color=color,
            width=width,
            arrows='to',
            physics=False
        )

    # --- 6. Save ---
    net.save_graph(output_file)
    print(f"Interactive visualization saved: {os.path.abspath(output_file)}")


visualize_lineage_interactive(subgraph, "med_missing_lineage.html")

Generating optimized interactive graph to med_missing_lineage.html...
Interactive visualization saved: /Users/kaushitha/Documents/APR/experiments/notebooks/coevolution/med_missing_lineage.html
